In [0]:
from pyspark.sql.functions import col, upper

bronze = "abfss://bronze@glcazurestorageproject.dfs.core.windows.net"
silver = "abfss://silver@glcazurestorageproject.dfs.core.windows.net"

df_user = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation", f"{silver}/DimUser/schema")
    .load(f"{bronze}/DimUser"))

df_user = df_user.withColumn("user_name", upper(col("user_name")))
df_user = df_user.drop("_rescued_data")

(df_user.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", f"{silver}/DimUser/checkpoint")
    .option("path", f"{silver}/DimUser/data")
    .trigger(availableNow=True)
    .toTable("glc_project.silver.dim_user"))

In [0]:
from pyspark.sql.functions import col, when

df_track = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation", f"{silver}/DimTrack/schema")
    .load(f"{bronze}/DimTrack"))

df_track = df_track.withColumn("duration_flag",
    when(col("duration_sec") < 150, "low")
    .when(col("duration_sec") < 300, "medium")
    .otherwise("high"))

df_track = df_track.drop("_rescued_data")

(df_track.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", f"{silver}/DimTrack/checkpoint")
    .option("path", f"{silver}/DimTrack/data")
    .trigger(availableNow=True)
    .toTable("glc_project.silver.dim_track"))

In [0]:
print(spark.read.format("parquet").load(f"{bronze}/DimArtist").columns)

In [0]:
# DimArtist
df_artist = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation", f"{silver}/DimArtist/schema")
    .load(f"{bronze}/DimArtist")
    .drop("_rescued_data"))

(df_artist.writeStream
    .format("delta").outputMode("append")
    .option("checkpointLocation", f"{silver}/DimArtist/checkpoint")
    .option("path", f"{silver}/DimArtist/data")
    .trigger(availableNow=True)
    .toTable("glc_project.silver.dim_artist"))

In [0]:
df_fact = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation", f"{silver}/FactStream/schema")
    .load(f"{bronze}/FactStream")
    .drop("_rescued_data"))

(df_fact.writeStream
    .format("delta").outputMode("append")
    .option("checkpointLocation", f"{silver}/FactStream/checkpoint")
    .option("path", f"{silver}/FactStream/data")
    .trigger(availableNow=True)
    .toTable("glc_project.silver.fact_stream"))

In [0]:
df_date = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation", f"{silver}/DimDate/schema")
    .load(f"{bronze}/DimDate")
    .drop("_rescued_data"))

(df_date.writeStream
    .format("delta").outputMode("append")
    .option("checkpointLocation", f"{silver}/DimDate/checkpoint")
    .option("path", f"{silver}/DimDate/data")
    .trigger(availableNow=True)
    .toTable("glc_project.silver.dim_date"))